# CorpDocWeave_Rag — Annotated Run Notebook (safe, no secrets)
_Last updated: 2025-11-07_

This copy adds guidance and removes hard‑coded secrets so you can push it to GitHub safely.

## How to run (TL;DR)
1. **Install package (editable) & deps** in your runtime.
2. **Provide secrets via environment** — _never_ hard‑code keys in cells. Use a `.env` file or Colab secrets.
3. **Choose a documents folder** (e.g. `data/mydoc/`) and ingest → build/update the Chroma index.
4. **Start the FastAPI app** (Uvicorn) for UI + API routes (e.g., `/ui`, `/ui/assistant`, `/kg/*`).
5. **Ask questions** via the minimal RAG chatbot or the UI; verify citations point to your chunks.
6. (Optional) **Build/preview a Knowledge Graph** slice for your entity.

## Your RAG model — short description
- **Ingestion & Chunking**: Reads PDFs/TXTs/etc., normalizes whitespace, splits into ~1200‑char chunks with overlap (~100).  
- **Embeddings**: Uses **OpenAI `text-embedding-3-small` (1536‑d)** when `OPENAI_API_KEY` is set; otherwise can fall back to local sentence transformers.
- **Vector Store**: **Chroma** persistent client (per‑tenant collection name `rag_<collection>`). Stores document text + metadata (`source`, `chunk_index`, `tenant_id`).
- **Retriever**: Top‑K similarity search by embedded query vector.
- **Generator**: **OpenAI chat model** (default `gpt-4o-mini`) instructed to answer **strictly from retrieved context**, with compact citations.
- **(Optional) KG**: Routes can build a small Neo4j‑backed subgraph from parsed claims/metrics; helpful for cross‑linking entities, metrics, and document claims.

## Secrets & safety
- Keep secrets in **environment variables** (`.env`, Colab secrets, or VM env). This notebook strips any inline keys it finds.
- `.gitignore` should exclude: `.env*`, `.ipynb_checkpoints/`, logs (`uvicorn.log`), and local vectorstore folders (`.chroma*/`).

## Troubleshooting quickies
- **`no such column: collections.topic`** → Chroma version mismatch; rebuild the index with a fresh `CHROMA_PATH` after upgrading Chroma.
- **`OPENAI_API_KEY not set`** → set it in the environment, not in code cells.
- **PDF text blank** → install `pymupdf`; fallback to `pypdf` if needed.

---


**Agentic RAG**

In [ ]:
from getpass import getpass
import os
os.environ["OPENAI_API_KEY"] = getpass("🔑 OpenAI API key: ")


In [ ]:
import os; print("Key set?", bool(os.getenv("OPENAI_API_KEY")))


In [ ]:
# @title 🔥 One-click: Launch AAA_Rag UI (API + KG + Chroma) with chroma_update toggle
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")  # ← put your real key

# Toggle: True = wipe & rebuild the Chroma collection; False = reuse existing
CHROMA_UPDATE = True  # ← switch as needed

# --- nothing below to edit ---
import os, sys, subprocess, pathlib
from google.colab import drive, output

REPO_DIR = "/content/drive/MyDrive/CorpDocWeave"
PYTHON = sys.executable

# 0) Mount Drive if needed
if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

assert pathlib.Path(REPO_DIR).exists(), f"Repo not found at {REPO_DIR}"

# 1) Env (key + defaults safe for Colab)
os.environ["OPENAI_API_KEY"]   = OPENAI_API_KEY
os.environ.setdefault("KG_BACKEND", "memory")
os.environ.setdefault("CHROMA_PATH", f"{REPO_DIR}/.chroma_oai1536")
os.environ.setdefault("COLLECTION_PREFIX", "rag")
os.environ.setdefault("COLLECTION", "mydoc_oai1536")
os.environ.setdefault("EMBED_MODEL", "text-embedding-3-small")  # 1536-dim
os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")               # used by launcher as CHAT_MODEL

CHROMA_PATH = os.environ["CHROMA_PATH"]
COLL = f"{os.environ['COLLECTION_PREFIX']}_{os.environ['COLLECTION']}"

# 2) Install package + runtime deps (idempotent)
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-q",
                "openai>=1.40.0", "chromadb>=0.5.4",
                "pymupdf", "pypdf", "python-docx", "requests",
                "hnswlib", "duckdb", "pysqlite3-binary"], check=True)

# 3) If requested, drop the existing Chroma collection (clean rebuild)
if CHROMA_UPDATE:
    import chromadb, shutil, os as _os
    print(f"🔁 chroma_update=True → deleting collection {COLL!r} at {CHROMA_PATH}")
    c = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        c.delete_collection(COLL)
        print("✔ collection deleted")
    except Exception as e:
        print(f"(info) delete_collection skipped or already absent: {e}")

# 4) Run the launcher once (starts API, builds KG, ingests docs)
env = os.environ.copy()
cmd = [PYTHON, "scripts/agentic_rag_launch.py", "--ui", "--build-kg", "--ingest"]
print("▶️  Launching…")
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

# 5) Open /ui in a new tab via Colab proxy
print("🔎 Probing Colab proxy …")
url = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("✅ Open this in a new tab:", url + "/ui")
output.eval_js(f"window.open('{url}/ui','_blank')")

# 6) Log hint
print("\n📄 Logs:", f"{REPO_DIR}/uvicorn.log")


**Knowledge Graph**

In [ ]:
# @title 🔥 One-click: Launch RAG+KG Assistant (/ui/assistant) with optional Chroma rebuild
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")  # ← put your real key
!pkill -f "uvicorn apps.api.main:app" || true
!rm -f /content/drive/MyDrive/CorpDocWeave/uvicorn.pid

# Toggle: True = wipe & rebuild Chroma collection; False = reuse if present
CHROMA_UPDATE = True                      # ← switch as needed

# --- nothing below to edit ---
import os, sys, subprocess, pathlib
from google.colab import drive, output

REPO_DIR = "/content/drive/MyDrive/CorpDocWeaveCorpDocWeave"
PYTHON = sys.executable

# 0) Mount Drive
if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
assert pathlib.Path(REPO_DIR).exists(), f"Repo not found at {REPO_DIR}"

# 1) Env
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ.setdefault("KG_BACKEND", "memory")  # change to postgres/neo4j via your KG launcher if desired
os.environ.setdefault("CHROMA_PATH", f"{REPO_DIR}/.chroma_oai1536")
os.environ.setdefault("COLLECTION_PREFIX", "rag")
os.environ.setdefault("COLLECTION", "mydoc_oai1536")
os.environ.setdefault("EMBED_MODEL", "text-embedding-3-small")  # 1536-dim
os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")               # used as CHAT_MODEL by the API

CHROMA_PATH = os.environ["CHROMA_PATH"]
COLL = f"{os.environ['COLLECTION_PREFIX']}_{os.environ['COLLECTION']}"

# 2) Deps
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-q",
                "openai>=1.40.0", "chromadb>=0.5.4",
                "pymupdf", "pypdf", "python-docx", "requests",
                "hnswlib", "duckdb", "pysqlite3-binary"], check=True)

# 3) Chroma collection handling (optional wipe)
do_ingest = CHROMA_UPDATE  # default: rebuild -> ingest
try:
    import chromadb
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.get_collection(COLL)     # exists?
        if CHROMA_UPDATE:
            print(f"🔁 chroma_update=True → deleting collection {COLL!r} at {CHROMA_PATH}")
            client.delete_collection(COLL)
            do_ingest = True
        else:
            # collection exists & reuse -> no ingest
            do_ingest = False
    except Exception:
        # collection missing -> must ingest
        do_ingest = True
except Exception as e:
    print("(info) chromadb check skipped:", e)
    # if we can't check, keep do_ingest as per toggle

# 4) Start API + build KG + ingest (only if needed)
env = os.environ.copy()
cmd = [PYTHON, "scripts/agentic_rag_launch.py", "--ui", "--build-kg"]
if do_ingest:
    cmd.append("--ingest")
print("▶️  Launching…", "with ingest" if do_ingest else "(no ingest)")
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

# 5) Open the RAG+KG Assistant UI
print("🔎 Probing Colab proxy …")
url = output.eval_js("google.colab.kernel.proxyPort(8000)")
assistant_url = url + "/ui/assistant"    # requires assistant route added to FastAPI
print("✅ Open this in a new tab:", assistant_url)
output.eval_js(f"window.open('{assistant_url}','_blank')")

print("\n📄 Logs:", f"{REPO_DIR}/uvicorn.log")
